<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/Qwen_3_TTS_By_(HunaarG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install with FlashAttention support
!pip install -U qwen-tts gradio huggingface_hub
!pip install flash-attn --no-build-isolation
!pip install pydub
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.4 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=abd8d9

clone , custom , podcast với giọng mặc định

In [ ]:
# import gradio as gr
# from qwen_tts import Qwen3TTSModel
# import torch, soundfile as sf, tempfile, gc
# import os

# # --- THÊM: Import thư viện xử lý âm thanh ---
# try:
#     from pydub import AudioSegment
# except ImportError:
#     print("⚠️ Chưa cài pydub. Vui lòng chạy '!pip install pydub' trước!")

# # ================= PERFORMANCE =================
# current_model = None
# current_model_type = None

# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.conv.fp32_precision = "tf32"
# torch.backends.cuda.matmul.fp32_precision = "tf32"

# print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")

# # ================= MODEL LOADER =================
# def load_model(t):
#     global current_model, current_model_type
#     if current_model_type == t:
#         return current_model

#     if current_model:
#         del current_model
#         gc.collect()
#         torch.cuda.empty_cache()

#     models = {
#         "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
#         "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
#         "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
#     }

#     current_model = Qwen3TTSModel.from_pretrained(
#         models[t],
#         torch_dtype=torch.float16,
#         device_map="cuda:0",
#         attn_implementation="sdpa"
#     )
#     current_model_type = t
#     return current_model

# # ================= FUNCTIONS (GỐC) =================
# def voice_clone(text, audio, transcript, fast):
#     if not text or not audio: return None
#     m = load_model("base")
#     p = m.create_voice_clone_prompt(
#         ref_audio=audio,
#         ref_text=None if fast else transcript,
#         x_vector_only_mode=fast
#     )
#     with torch.inference_mode():
#         w, sr = m.generate_voice_clone(text=text, voice_clone_prompt=p)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def custom_voice(text, voice, inst):
#     if not text: return None
#     m = load_model("custom")
#     with torch.inference_mode():
#         w, sr = m.generate_custom_voice(
#             text=text,
#             speaker=voice,
#             instruct=inst if inst.strip() else None
#         )
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def voice_design(text, desc):
#     if not text or not desc: return None
#     m = load_model("design")
#     with torch.inference_mode():
#         w, sr = m.generate_voice_design(text=text, instruct=desc)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# # ================= FUNCTION MỚI: PODCAST MAKER =================
# def generate_podcast(script_text):
#     if not script_text: return None

#     # 1. Tải model Custom (Chỉ tải 1 lần)
#     m = load_model("custom")

#     # 2. Cấu hình nhân vật (Bạn có thể sửa Instruction ở đây)
#     CHAR_CONFIG = {
#         "Eric":   {"voice": "eric",   "inst": "Professional male teacher, deep warm voice, educational tone"},
#         "Serena": {"voice": "serena", "inst": "Friendly female host, energetic, happy tone"},
#         "Ryan":   {"voice": "ryan",   "inst": "British male voice, professional and calm"},
#         "Vivian": {"voice": "vivian", "inst": "Young female voice, fast and fun"}
#     }

#     final_audio = AudioSegment.silent(duration=500) # Bắt đầu với 0.5s im lặng
#     gap = AudioSegment.silent(duration=400) # Nghỉ 0.4s giữa các câu

#     lines = script_text.strip().split('\n')

#     # 3. Duyệt từng dòng kịch bản
#     for line in lines:
#         if ":" in line:
#             # Tách tên và lời thoại (VD: "Eric: Hello world")
#             name, text = line.split(":", 1)
#             name = name.strip()
#             text = text.strip()

#             # Kiểm tra xem tên có trong danh sách không
#             config = CHAR_CONFIG.get(name)

#             if config and text:
#                 print(f"🎙️ Đang tạo giọng cho {name}...")
#                 with torch.inference_mode():
#                     w, sr = m.generate_custom_voice(
#                         text=text,
#                         speaker=config["voice"],
#                         instruct=config["inst"]
#                     )

#                 # Lưu file tạm
#                 temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#                 sf.write(temp_wav.name, w[0], sr)

#                 # Đọc file bằng Pydub và ghép vào
#                 segment = AudioSegment.from_wav(temp_wav.name)
#                 final_audio += segment + gap

#                 # Xóa file tạm cho nhẹ máy
#                 os.unlink(temp_wav.name)
#             else:
#                 print(f"⚠️ Không tìm thấy cấu hình cho nhân vật: {name} (Hoặc lời thoại trống)")

#     # 4. Xuất file cuối cùng
#     output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
#     final_audio.export(output_path, format="mp3")
#     print("✅ Đã ghép xong Podcast!")
#     return output_path

# # ================= STRONG UI CSS =================
# css = """
# body{
# background:radial-gradient(circle at top,#eef2ff,#e0e7ff,#f8fafc);
# font-family:Inter,system-ui;
# }
# .hero{
# padding:26px;border-radius:20px;
# background:linear-gradient(135deg,#4f46e5,#7c3aed);
# color:white;text-align:center;
# box-shadow:0 20px 50px rgba(0,0,0,.25);
# margin-bottom:25px;
# }
# .hero h1{font-size:28px;margin-bottom:6px}
# .hero p{opacity:.9}
# .socials{display:flex;gap:14px;justify-content:center;flex-wrap:wrap;margin:22px 0}
# .btn{
# padding:12px 22px;border-radius:999px;
# font-weight:600;text-decoration:none;
# color:white;display:flex;gap:8px;
# align-items:center;transition:.35s;
# box-shadow:0 10px 30px rgba(0,0,0,.2)
# }
# .btn:hover{transform:translateY(-4px) scale(1.04)}
# .yt{background:#ff0000}
# .ig{background:linear-gradient(45deg,#f58529,#dd2a7b,#8134af)}
# .wa{background:#25D366}
# .panel{
# background:rgba(255,255,255,.7);
# backdrop-filter:blur(14px);
# border-radius:20px;
# padding:20px;
# box-shadow:0 15px 40px rgba(0,0,0,.15)
# }
# .footer{
# margin-top:30px;text-align:center;
# font-size:14px;color:#444
# }
# """

# # ================= UI =================
# with gr.Blocks(title="Qwen3-TTS | Hunaar Ansari", css=css) as demo:

#     gr.HTML("""
#     <div class="hero">
#         <h1>🎙️ Qwen3-TTS Voice Studio</h1>
#         <p>Professional AI Voice Cloning • Custom Voices • Voice Design</p>
#         <p><b>Created by Hunaar Ansari (Modified for Podcast)</b></p>
#     </div>
#     """)

#     gr.HTML("""
#     <div class="socials">
#         <a class="btn yt" href="http://www.youtube.com/@Hunaarg" target="_blank">YouTube</a>
#         <a class="btn ig" href="https://www.instagram.com/hunaar_ansari/" target="_blank">Instagram</a>
#         <a class="btn wa" href="https://whatsapp.com/channel/0029VabTEup4dTnKxHsYP01W" target="_blank">
#             WhatsApp Channel
#         </a>
#     </div>
#     """)

#     with gr.Tab("🎤 Voice Clone"):
#         with gr.Group(elem_classes="panel"):
#             t = gr.Textbox(lines=4, label="Text")
#             a = gr.Audio(type="filepath", label="Reference Audio")
#             tr = gr.Textbox(lines=2, label="Transcript (optional)")
#             f = gr.Checkbox(value=True, label="Fast Mode")
#             o = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_clone,[t,a,tr,f],o)

#     with gr.Tab("🎭 Custom Voice"):
#         with gr.Group(elem_classes="panel"):
#             t2 = gr.Textbox(lines=4)
#             v = gr.Dropdown(
#                 ["serena","vivian","ono_anna","sohee","aiden","dylan","eric","ryan","uncle_fu"],
#                 value="serena"
#             )
#             i = gr.Textbox(lines=2)
#             o2 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 custom_voice,[t2,v,i],o2)

#     with gr.Tab("🎨 Voice Design"):
#         with gr.Group(elem_classes="panel"):
#             t3 = gr.Textbox(lines=4)
#             d = gr.Textbox(lines=3)
#             o3 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_design,[t3,d],o3)

#     # --- TAB MỚI: PODCAST MODE ---
#     with gr.Tab("🎬 Podcast Mode (Kịch bản)"):
#         with gr.Group(elem_classes="panel"):
#             gr.Markdown("### Nhập kịch bản theo mẫu: `Tên: Lời thoại`")
#             gr.Markdown("Nhân vật hỗ trợ: **Eric** (Nam), **Serena** (Nữ), **Ryan**, **Vivian**")

#             script_input = gr.Textbox(
#                 lines=10,
#                 label="Kịch bản (Script)",
#                 placeholder="Eric: Hello everyone, welcome to the show.\nSerena: Hi Eric! I am so happy to be here.\nEric: Today we talk about AI."
#             )
#             podcast_output = gr.Audio(label="File Podcast Hoàn Chỉnh")

#             gr.Button("🎬 TẠO PODCAST & GHÉP FILE", variant="primary").click(
#                 generate_podcast, [script_input], podcast_output
#             )

#     gr.HTML("""
#     <div class="footer">
#         🚀 Built by <b>Hunaar Ansari</b><br>
#         Follow • Join • Learn AI Voice Technology
#     </div>
#     """)

# print("🔥 Qwen3-TTS Ultra UI | Hunaar Ansari")
# demo.launch(share=True, debug=True, theme=gr.themes.Soft())

In [ ]:
import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch, soundfile as sf, gc, os, re
import uuid
from pydub import AudioSegment

# ================= CẤU HÌNH & THƯ VIỆN =================
TEMP_DIR = "/content/gradio_output"
os.makedirs(TEMP_DIR, exist_ok=True)

# Performance Settings
torch.backends.cudnn.benchmark = True
print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")

# Biến toàn cục
current_model = None
current_model_type = None

# ================= 1. CÔNG CỤ XỬ LÝ VĂN BẢN THÔNG MINH =================
def smart_split_text(text, max_chars=250):
    """
    Cắt văn bản thành các đoạn nhỏ an toàn cho AI, nhưng không cắt giữa câu.
    """
    # Bước 1: Tách các đoạn văn (Paragraphs)
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]

    final_chunks = []

    for para in paragraphs:
        # Bước 2: Tách câu trong đoạn văn (dựa vào dấu kết câu)
        sentences = re.split(r'(?<=[.!?])\s+', para)

        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) < max_chars:
                current_chunk += sentence + " "
            else:
                if current_chunk: final_chunks.append(current_chunk.strip())
                current_chunk = sentence + " "
        if current_chunk: final_chunks.append(current_chunk.strip())

        # Đánh dấu kết thúc đoạn văn bằng ký tự đặc biệt để chèn khoảng lặng dài hơn
        final_chunks.append("<PARA_BREAK>")

    return final_chunks

# ================= 2. MODEL LOADER =================
def load_model(target_type="base"):
    global current_model, current_model_type
    if current_model_type == target_type and current_model is not None:
        return current_model

    print(f"🔄 Đang tải Model: {target_type}...")
    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    models = {
        "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
        "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
    }

    current_model = Qwen3TTSModel.from_pretrained(
        models[target_type],
        torch_dtype=torch.float16,
        device_map="cuda:0",
        attn_implementation="sdpa"
    )
    current_model_type = target_type
    return current_model

def get_temp_path(ext=".wav"):
    return os.path.join(TEMP_DIR, f"{uuid.uuid4()}{ext}")

# ================= 3. HÀM CLONE DÀI (AUTO MERGE) =================
def generate_long_audio(text, audio_ref, transcript_ref, fast_mode, progress=gr.Progress()):
    if not text or not audio_ref: return None

    progress(0.0, desc="🚀 Khởi động hệ thống...")
    m = load_model("base")

    # Tạo AudioSegment tổng (Trống)
    full_audio = AudioSegment.silent(duration=500)

    # Cấu hình khoảng lặng (Silence)
    short_silence = AudioSegment.silent(duration=200) # Nghỉ giữa các câu (0.2s)
    long_silence = AudioSegment.silent(duration=800)  # Nghỉ giữa các đoạn văn (0.8s) - Nghe sâu lắng hơn

    # Cắt nhỏ văn bản
    chunks = smart_split_text(text)
    total_chunks = len(chunks)

    # Tạo Prompt Clone (Chạy 1 lần dùng nhiều lần để tiết kiệm thời gian)
    progress(0.1, desc="🧠 Đang học giọng mẫu (Chỉ làm 1 lần)...")
    voice_prompt = m.create_voice_clone_prompt(
        ref_audio=audio_ref,
        ref_text=None if fast_mode else transcript_ref,
        x_vector_only_mode=fast_mode
    )

    print(f"📝 Đã chia văn bản thành {total_chunks} phần nhỏ. Bắt đầu xử lý...")

    for i, chunk in enumerate(chunks):
        if chunk == "<PARA_BREAK>":
            # Nếu gặp dấu hiệu hết đoạn văn -> Chèn khoảng lặng dài
            full_audio += long_silence
            continue

        progress((i / total_chunks), desc=f"🎙️ Đang đọc đoạn {i+1}/{total_chunks}...")

        try:
            with torch.inference_mode():
                # Generate đoạn ngắn
                w, sr = m.generate_voice_clone(text=chunk, voice_clone_prompt=voice_prompt)

            # Lưu file tạm
            temp_path = get_temp_path()
            sf.write(temp_path, w[0], sr)

            # Load lại bằng Pydub và nối vào file tổng
            segment = AudioSegment.from_wav(temp_path)
            full_audio += segment + short_silence # Nối thêm khoảng nghỉ ngắn

            # Dọn dẹp ngay lập tức
            os.remove(temp_path)

        except Exception as e:
            print(f"⚠️ Lỗi ở đoạn: {chunk[:20]}... -> {str(e)}")
            continue

    # Xuất file cuối cùng
    progress(0.9, desc="💾 Đang ghép file và xuất MP3...")
    final_path = get_temp_path(".mp3")
    full_audio.export(final_path, format="mp3")

    print(f"✅ Hoàn tất! File dài: {len(full_audio)/1000} giây")
    return final_path

# ================= 4. GIAO DIỆN (UI) =================
css = """
body{background:radial-gradient(circle at top,#1e1e2e,#2d2b55); color: white; font-family: sans-serif;}
.hero{text-align:center; padding: 20px; background: #4f46e5; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 30px rgba(0,0,0,0.5);}
.btn-primary{background: linear-gradient(90deg, #ff8a00, #e52e71) !important; color: white !important; font-weight: bold; border: none;}
"""

with gr.Blocks(title="Qwen3-TTS Long Audio Maker", css=css) as demo:
    gr.HTML("""
    <div class="hero">
        <h1>🎞️ LONG FORM VOICE CLONER</h1>
        <p>Paste 2000 từ vào đây -> Nhận về 1 file MP3 duy nhất (15 phút)</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            ref_audio = gr.Audio(label="1. File Giọng Mẫu (Reference Audio)", type="filepath")
            ref_text = gr.Textbox(label="2. Lời thoại của File mẫu (Transcript)", lines=2, placeholder="Gõ đúng những gì file mẫu nói...")
            fast_mode = gr.Checkbox(label="Fast Mode (Khuyên dùng: Tắt để hay hơn)", value=False)

        with gr.Column(scale=2):
            input_text = gr.Textbox(label="3. Kịch bản Video (Paste hết vào đây)", lines=15, placeholder="Paste cả kịch bản dài vào đây. Hệ thống tự cắt và ghép.")

    btn_generate = gr.Button("🚀 TẠO AUDIO DÀI (AUTO MERGE)", elem_classes="btn-primary")
    output_audio = gr.Audio(label="4. Kết quả (Tải về tại đây)", type="filepath")

    btn_generate.click(
        generate_long_audio,
        inputs=[input_text, ref_audio, ref_text, fast_mode],
        outputs=[output_audio]
    )

print("🔥 Hệ thống đã sẵn sàng! Mở link Public URL bên dưới...")
demo.queue().launch(share=True, debug=True)